In [5]:
import aiohttp
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from settings import Settings

settings = Settings()
settings = settings.entsoe_api_key

BASE_URL = "https://web-api.tp.entsoe.eu/api"
SWISS_DOMAIN = "10YCH-SWISSGRIDZ"


async def fetch_entsoe_to_parquet(
    api_key: str,
    start_year: int,
    end_year: int,
    save_dir: str | Path = Path.cwd().parent / "data" / "raw",
    file_name: str = "swiss_load_full.parquet",
):
    """
    Fetch Swiss ENTSO-E load year by year,
    parse immediately,
    concatenate all years,
    save one parquet.
    """

    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    all_rows = []

    async with aiohttp.ClientSession() as session:

        for year in range(start_year, end_year + 1):

            start = f"{year}01010000"
            end = f"{year + 1}01010000"

            params = {
                "documentType": "A65",
                "processType": "A16",
                "outBiddingZone_Domain": SWISS_DOMAIN,
                "periodStart": start,
                "periodEnd": end,
                "securityToken": api_key,
            }

            async with session.get(BASE_URL, params=params) as response:
                response.raise_for_status()

                xml_text = await response.text()

                print(f"Fetched {year}")

                root = ET.fromstring(xml_text)

                for period in root.iter():

                    if not period.tag.endswith("Period"):
                        continue

                    start_dt = None
                    resolution = None

                    for node in period.iter():

                        if node.tag.endswith("start"):
                            start_dt = pd.Timestamp(node.text, tz="UTC")

                        elif node.tag.endswith("resolution"):
                            resolution = node.text

                    if start_dt is None:
                        continue

                    delta = (
                        pd.Timedelta(hours=1)
                    )

                    for point in period:

                        if point.tag.endswith("Point"):

                            vals = {
                                x.tag.split("}")[-1]: x.text
                                for x in point
                            }

                            ts = start_dt + (
                                int(vals["position"]) - 1
                            ) * delta

                            all_rows.append(
                                {
                                    "timestamp_utc": ts,
                                    "load_mw": float(vals["quantity"]),
                                }
                            )

    df = (
        pd.DataFrame(all_rows)
        .drop_duplicates(subset="timestamp_utc")
        .sort_values("timestamp_utc")
        .reset_index(drop=True)
    )

    out_file = save_path / file_name
    df.to_parquet(out_file, index=False)

    print(f"Saved: {out_file}")
    print(df.shape)

    return df

In [8]:
df = await fetch_entsoe_to_parquet(
    api_key=settings,
    start_year=2020,
    end_year=2026
)

Fetched 2020
Fetched 2021
Fetched 2022
Fetched 2023
Fetched 2024
Fetched 2025
Fetched 2026
Saved: /home/alvard/projects/load_forecasting/data/raw/swiss_load_full.parquet
(55158, 2)
